## Structure Hierarchical Classifier

In [30]:
import torch
import torch.nn as nn

class HierarchicalClassifier(nn.Module):
    def __init__(self, binary_clf: nn.Module, multi_clf: nn.Module, device="cpu"):
        """
        Args:
            binary_clf (nn.Module): Binary classifier (N vs F).
            multi_clf (nn.Module): Multi-class classifier (B, I, O).
            device (str): Device to run the model.
        """
        super().__init__()
        self.binary_clf = binary_clf.to(device)
        self.multi_clf = multi_clf.to(device)
        self.device = device

        # Map final labels: 0=N, 1=B, 2=I, 3=O
        self.final_classes = {0: "N", 1: "B", 2: "I", 3: "O"}

    def forward(self, x):
        """
        Forward pass through the hierarchical classifier.
        
        Args:
            x (torch.Tensor): Input batch [batch_size, features].

        Returns:
            torch.Tensor: Final predictions (0=N, 1=B, 2=I, 3=O).
        """
        x = x.to(self.device)

        # Step 1: Binary classification (N=0, F=1)
        bin_out = self.binary_clf(x)  
        bin_pred = torch.argmax(bin_out, dim=1)

        final_pred = []
        for i, pred in enumerate(bin_pred):
            if pred.item() == 0:  
                # Class N (normal bearing)
                final_pred.append(0)
            else:
                # Step 2: Multi-class classification (B=1, I=2, O=3)
                multi_out = self.multi_clf(x[i].unsqueeze(0))
                multi_pred = torch.argmax(multi_out, dim=1).item()
                final_pred.append(multi_pred + 1)  # shift to {1,2,3}
        
        return torch.tensor(final_pred, device=self.device)

    def predict(self, x):
        """Convenience method for prediction with labels."""
        preds = self.forward(x)
        return [self.final_classes[p.item()] for p in preds]


In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class HierarchicalClassifierProb(nn.Module):
    """
    Combina dois classificadores:
      - binary_clf: logits para [N, F] (shape: [N, 2])
      - multi_clf : logits para [B, I, O] (shape: [N, 3])

    Forward retorna log-probabilidades para 4 classes na ordem [N, B, I, O],
    compatíveis com nn.NLLLoss (target: 0=N, 1=B, 2=I, 3=O).
    """
    def __init__(self, binary_clf: nn.Module, multi_clf: nn.Module, device="cpu"):
        super().__init__()
        self.binary_clf = binary_clf.to(device)
        self.multi_clf  = multi_clf.to(device)
        self.device = device

    def forward(self, x):
        x = x.to(self.device)

        # Logits dos dois classificadores
        bin_logits   = self.binary_clf(x)         # [N, 2] => [N, (N,F)]
        multi_logits = self.multi_clf(x)          # [N, 3] => [N, (B,I,O)]

        # Convertemos para log-probabilidades (log-softmax)
        bin_logp   = F.log_softmax(bin_logits,   dim=1)  # [N,2]
        multi_logp = F.log_softmax(multi_logits, dim=1)  # [N,3]

        # Combinação hierárquica em log-espaco
        logP_N = bin_logp[:, 0]                                  # P(N)
        logP_B = bin_logp[:, 1] + multi_logp[:, 0]               # P(F)*P(B|F)
        logP_I = bin_logp[:, 1] + multi_logp[:, 1]               # P(F)*P(I|F)
        logP_O = bin_logp[:, 1] + multi_logp[:, 2]               # P(F)*P(O|F)

        # Empilha em [N, B, I, O] => shape [N,4]
        logP_4 = torch.stack([logP_N, logP_B, logP_I, logP_O], dim=1)
        return logP_4  # use NLLLoss

    @torch.no_grad()
    def predict(self, x):
        logp = self.forward(x)
        preds = torch.argmax(logp, dim=1)  # 0=N,1=B,2=I,3=O
        idx2label = {0: "N", 1: "B", 2: "I", 3: "O"}
        return [idx2label[i.item()] for i in preds]


In [ ]:
# Suponha que você já tenha treinado:
# binary_clf -> classificador N vs F
# multi_clf  -> classificador B/I/O

hier_clf = HierarchicalClassifier(binary_clf, multi_clf, device="cuda")

# Exemplo de inferência
x = torch.randn(8, 250).to("cuda")  # batch de 8 amostras (250 pontos cada)
preds = hier_clf.predict(x)
print(preds)  # -> ['N', 'B', 'I', 'O', ...]


## Dataset

In [ ]:
import os
from glob import glob
import numpy as np
import torch
from torch.utils.data import Dataset

# Label maps
LABELS = {
    "bin": {"N": 0, "F": 1},                # non-fault vs fault
    "bio": {"B": 0, "I": 1, "O": 2},        # only fault types
    "bino": {"N": 0, "B": 1, "I": 2, "O": 3} # all classes together
}

class BearingNPYDataset(Dataset):
    """
    Loads .npy 1D signals organized as:
      root/bin/(N_1_0.npy,...,F_2_1.npy)   -> task="bin"
      root/bio/(B_5_2.npy,...,I_9_5.npy,...) -> task="bio"
      root/bino/(N_1_0.npy,B_2_1.npy,...) -> task="bino"
    Label is inferred from the first char in filename.
    """

    def __init__(self, root_dir, task="bin", transform=None, target_transform=None, return_path=False):
        self.root_dir = root_dir
        self.task = task.lower().strip()
        if self.task not in LABELS:
            raise ValueError("task must be 'bin', 'bio' or 'bino'")
        self.map = LABELS[self.task]
        self.transform = transform
        self.target_transform = target_transform
        self.return_path = return_path

        # collect files
        paths = sorted(glob(os.path.join(root_dir, "*.npy")))
        if not paths:
            raise FileNotFoundError("No .npy files found in %s" % root_dir)

        # keep only files whose first char is in label map
        self.samples = [p for p in paths if os.path.basename(p)[0] in self.map]
        if not self.samples:
            raise FileNotFoundError("No valid labeled files (by prefix) in %s" % root_dir)

        # store class names in index order
        inv = sorted(self.map.items(), key=lambda x: x[1])
        self.classes = [k for k, _ in inv]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path = self.samples[idx]
        x = np.load(path)                 # expects 1D numpy array
        x = torch.from_numpy(x).float()   # to float32 tensor
        if x.ndim == 1:
            x = x.unsqueeze(0)            # shape [1, L] for Conv1d

        y_char = os.path.basename(path)[0]
        y = torch.tensor(self.map[y_char], dtype=torch.long)

        if self.transform is not None:
            x = self.transform(x)
        if self.target_transform is not None:
            y = self.target_transform(y)

        if self.return_path:
            return x, y, path
        return x, y


In [2]:
# ---------- Exemplo -----------------

root = "data/processed/uored_hierarchical/setup_1"
# Binário
bin_ds = BearingNPYDataset(f"{root}/bin", task="bin")
# Multiclasse
bio_ds = BearingNPYDataset(f"{root}/bio", task="bio")
# Teste
te_bin_ds = BearingNPYDataset(f"{root}/test", task="bin")

# DataLoader
from torch.utils.data import DataLoader
bin_loader = DataLoader(bin_ds, batch_size=64, shuffle=True, num_workers=0)
bio_loader = DataLoader(bio_ds, batch_size=64, shuffle=True, num_workers=0)
te_bin_loader = DataLoader(te_bin_ds, batch_size=64, shuffle=False, num_workers=0)

# Acessar nomes das classes
print(bin_ds.classes)  # ['N', 'F']
print(te_bin_ds.classes)
print(bio_ds.classes)  # ['B', 'I', 'O']


['N', 'F']
['N', 'F']
['B', 'I', 'O']


## Training

In [3]:
import os
import math
import torch
from torch import nn
from torch.optim import Adam, SGD
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR, OneCycleLR
from collections import defaultdict
from time import time

def _accuracy(logits, targets):
    preds = torch.argmax(logits, dim=1)
    return (preds == targets).float().mean().item()

def _build_optimizer(model, opt_cfg):
    name = (opt_cfg.get('name') or 'adam').lower()
    lr = opt_cfg.get('lr', 1e-3)
    wd = opt_cfg.get('weight_decay', 0.0)
    if name == 'sgd':
        momentum = opt_cfg.get('momentum', 0.9)
        nesterov = opt_cfg.get('nesterov', True)
        return SGD(model.parameters(), lr=lr, weight_decay=wd, momentum=momentum, nesterov=nesterov)
    # default adam
    betas = opt_cfg.get('betas', (0.9, 0.999))
    eps = opt_cfg.get('eps', 1e-8)
    return Adam(model.parameters(), lr=lr, weight_decay=wd, betas=betas, eps=eps)

def _build_scheduler(optimizer, sch_cfg, train_loader, epochs, has_val):
    """Returns (scheduler, step_on) where step_on in {'epoch','iteration','val_metric','none'}"""
    if not sch_cfg or sch_cfg.get('name') is None:
        return None, 'none'
    name = sch_cfg['name'].lower()
    if name in ['plateau', 'reduceonplateau', 'reduce_lr_on_plateau']:
        # Step when validation accuracy plateaus (needs val)
        if not has_val:
            return None, 'none'
        factor = sch_cfg.get('factor', 0.5)
        patience = sch_cfg.get('patience', 5)
        cooldown = sch_cfg.get('cooldown', 0)
        min_lr = sch_cfg.get('min_lr', 1e-6)
        verbose = sch_cfg.get('verbose', True)
        return ReduceLROnPlateau(optimizer, mode='max', factor=factor, patience=patience,
                                 cooldown=cooldown, min_lr=min_lr, verbose=verbose), 'val_metric'
    if name in ['cosine', 'cosineannealinglr']:
        T_max = sch_cfg.get('t_max', epochs)
        eta_min = sch_cfg.get('eta_min', 0.0)
        return CosineAnnealingLR(optimizer, T_max=T_max, eta_min=eta_min), 'epoch'
    if name in ['onecycle', 'onecyclelr']:
        # Needs steps_per_epoch
        max_lr = sch_cfg.get('max_lr', optimizer.param_groups[0]['lr'])
        steps_per_epoch = sch_cfg.get('steps_per_epoch', len(train_loader))
        pct_start = sch_cfg.get('pct_start', 0.3)
        div_factor = sch_cfg.get('div_factor', 25.0)
        final_div_factor = sch_cfg.get('final_div_factor', 1e4)
        anneal_strategy = sch_cfg.get('anneal_strategy', 'cos')
        return OneCycleLR(optimizer, max_lr=max_lr, epochs=epochs, steps_per_epoch=steps_per_epoch,
                          pct_start=pct_start, div_factor=div_factor, final_div_factor=final_div_factor,
                          anneal_strategy=anneal_strategy), 'iteration'
    return None, 'none'

def _save_checkpoint(state, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(state, path)

def train_model(config):
    """
    Expected keys in `config` (all optional except model, train_loader):
      - model: torch.nn.Module
      - train_loader: DataLoader
      - val_loader: DataLoader or None
      - device: 'cuda' or 'cpu' (default: auto)
      - epochs: int (default 20)
      - criterion: loss function (default CrossEntropyLoss)
      - optimizer: dict -> {name: 'adam'|'sgd', lr, weight_decay, ...}
      - scheduler: dict -> {name: 'plateau'|'cosine'|'onecycle', ...}
        * If 'plateau', it will monitor best validation accuracy.
      - amp: bool (default True)
      - grad_clip: float or None (default None)
      - early_stopping: dict or None -> {patience: int, min_delta: float}
      - checkpoint_dir: str (default 'checkpoints')
      - checkpoint_name: str (default 'best.pt')
      - save_every_epoch: bool (default False)
      - log_interval: int (default 50)
      - eval_interval: int (default 1)  # validate every N epochs
    Returns:
      history (dict) and best_ckpt_path (str or None)
    """
    model = config['model']
    train_loader = config['train_loader']
    val_loader = config.get('val_loader')
    device = config.get('device') or ('cuda' if torch.cuda.is_available() else 'cpu')
    epochs = config.get('epochs', 20)
    criterion = config.get('criterion') or nn.CrossEntropyLoss()
    amp_enabled = config.get('amp', True)
    grad_clip = config.get('grad_clip', None)
    early_cfg = config.get('early_stopping') or {}
    early_patience = early_cfg.get('patience', None)
    early_min_delta = early_cfg.get('min_delta', 0.0)
    ckpt_dir = config.get('checkpoint_dir', 'checkpoints')
    ckpt_name = config.get('checkpoint_name', 'best.pt')
    save_every_epoch = config.get('save_every_epoch', False)
    log_interval = config.get('log_interval', 50)
    eval_interval = max(1, config.get('eval_interval', 1))

    model.to(device)
    optimizer = _build_optimizer(model, config.get('optimizer', {}))
    scheduler, sched_step_on = _build_scheduler(optimizer, config.get('scheduler', {}),
                                                train_loader, epochs, has_val=val_loader is not None)
    scaler = torch.cuda.amp.GradScaler(enabled=(amp_enabled and device == 'cuda'))

    history = defaultdict(list)
    best_val_acc = -float('inf')
    best_epoch = -1
    no_improve_epochs = 0
    best_ckpt_path = None
    start_time = time()

    for epoch in range(1, epochs + 1):
        # ---- Train ----
        model.train()
        run_loss = 0.0
        run_acc = 0.0
        n_batches = 0

        for i, batch in enumerate(train_loader, 1):
            if isinstance(batch, (list, tuple)) and len(batch) == 2:
                x, y = batch
            else:
                # Expect dict with 'inputs' and 'targets'
                x, y = batch['inputs'], batch['targets']

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if amp_enabled and device == 'cuda':
                with torch.amp.autocast('cuda'):
                    logits = model(x)
                    loss = criterion(logits, y)
                scaler.scale(loss).backward()
                if grad_clip is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(x)
                loss = criterion(logits, y)
                loss.backward()
                if grad_clip is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            acc = _accuracy(logits, y)
            run_loss += loss.item()
            run_acc += acc
            n_batches += 1

            if scheduler is not None and sched_step_on == 'iteration':
                scheduler.step()

            if log_interval and (i % log_interval == 0):
                print(f"Epoch {epoch:03d} | step {i:04d}/{len(train_loader)} | "
                      f"loss {loss.item():.4f} | acc {acc:.4f} | lr {optimizer.param_groups[0]['lr']:.6f}")

        tr_loss = run_loss / max(1, n_batches)
        tr_acc = run_acc / max(1, n_batches)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)

        # ---- Validate ----
        do_validate = (val_loader is not None) and (epoch % eval_interval == 0)
        if do_validate:
            model.eval()
            val_loss = 0.0
            val_acc = 0.0
            vb = 0
            with torch.no_grad():
                for batch in val_loader:
                    if isinstance(batch, (list, tuple)) and len(batch) == 2:
                        x, y = batch
                    else:
                        x, y = batch['inputs'], batch['targets']
                    x = x.to(device, non_blocking=True)
                    y = y.to(device, non_blocking=True)

                    if amp_enabled and device == 'cuda':
                        with torch.amp.autocast('cuda'):
                            logits = model(x)
                            loss = criterion(logits, y)
                    else:
                        logits = model(x)
                        loss = criterion(logits, y)

                    val_loss += loss.item()
                    val_acc += _accuracy(logits, y)
                    vb += 1
            val_loss /= max(1, vb)
            val_acc /= max(1, vb)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)

            # ---- Scheduler on validation metric (best val acc) ----
            if scheduler is not None and sched_step_on == 'val_metric':
                scheduler.step(val_acc)

            # ---- Checkpoint on improvement ----
            improved = val_acc > (best_val_acc + early_min_delta)
            if improved:
                best_val_acc = val_acc
                best_epoch = epoch
                best_ckpt_path = os.path.join(ckpt_dir, ckpt_name)
                _save_checkpoint({
                    'epoch': epoch,
                    'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'scaler_state': scaler.state_dict() if amp_enabled and device == 'cuda' else None,
                    'best_val_acc': best_val_acc,
                    'config': config
                }, best_ckpt_path)
                no_improve_epochs = 0
                print(f"[Best] epoch {epoch} | val_acc {val_acc:.4f} | saved -> {best_ckpt_path}")
            else:
                no_improve_epochs += 1

            # ---- Early stopping (based on val acc) ----
            if early_patience is not None and no_improve_epochs >= early_patience:
                print(f"[EarlyStopping] No improvement for {no_improve_epochs} epochs "
                      f"(best epoch {best_epoch}, best val_acc {best_val_acc:.4f}).")
                break

            print(f"Epoch {epoch:03d} | train: loss {tr_loss:.4f}, acc {tr_acc:.4f} | "
                  f"val: loss {val_loss:.4f}, acc {val_acc:.4f} | "
                  f"lr {optimizer.param_groups[0]['lr']:.6f}")
        else:
            # No validation this epoch
            if scheduler is not None and sched_step_on == 'epoch':
                scheduler.step()
            print(f"Epoch {epoch:03d} | train: loss {tr_loss:.4f}, acc {tr_acc:.4f} | "
                  f"lr {optimizer.param_groups[0]['lr']:.6f}")

        # Optional: save every epoch (rolling checkpoints)
        if save_every_epoch:
            path = os.path.join(ckpt_dir, f"epoch_{epoch:03d}.pt")
            _save_checkpoint({'epoch': epoch, 'model_state': model.state_dict()}, path)

    elapsed = time() - start_time
    print(f"Training finished in {elapsed/60:.1f} min. Best val_acc={best_val_acc:.4f} at epoch {best_epoch}.")
    return dict(history), best_ckpt_path


In [27]:
# Exemplo
from src.models import BearingCNN1D

val_ds = BearingNPYDataset(f"{root}/bin_val", task="bin")
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

bin_model = BearingCNN1D(num_classes=2)
cfg = {
    'model': bin_model,                     # nn.Module já criado
    'train_loader': bin_loader,          # DataLoader
    'val_loader': None,              # ou None
    'epochs': 30,
    'optimizer': {'name': 'adam', 'lr': 1e-4, 'weight_decay': 1e-4},
    'scheduler': {'name': 'plateau', 'factor': 0.5, 'patience': 5, 'min_lr': 1e-6},
    'criterion': torch.nn.CrossEntropyLoss(),
    'amp': True,
    'grad_clip': 1.0,
    'early_stopping': {'patience': 30, 'min_delta': 0.0},
    'checkpoint_dir': 'checkpoints',
    'checkpoint_name': 'best.pt',
    'save_every_epoch': False,
    'log_interval': 50,
    'eval_interval': 1,
}
history, best_path = train_model(cfg)


/tmp/ipykernel_3149089/1975858508.py:108: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp_enabled and device == 'cuda'))


Epoch 001 | step 0050/60 | loss 0.2573 | acc 0.9375 | lr 0.000100
Epoch 001 | train: loss 0.3053, acc 0.9219 | lr 0.000100
Epoch 002 | step 0050/60 | loss 0.2385 | acc 0.9375 | lr 0.000100
Epoch 002 | train: loss 0.2589, acc 0.9344 | lr 0.000100
Epoch 003 | step 0050/60 | loss 0.2227 | acc 0.9531 | lr 0.000100
Epoch 003 | train: loss 0.2322, acc 0.9466 | lr 0.000100
Epoch 004 | step 0050/60 | loss 0.2606 | acc 0.9062 | lr 0.000100
Epoch 004 | train: loss 0.2141, acc 0.9518 | lr 0.000100
Epoch 005 | step 0050/60 | loss 0.1662 | acc 0.9688 | lr 0.000100
Epoch 005 | train: loss 0.1916, acc 0.9667 | lr 0.000100
Epoch 006 | step 0050/60 | loss 0.1688 | acc 0.9375 | lr 0.000100
Epoch 006 | train: loss 0.1727, acc 0.9711 | lr 0.000100
Epoch 007 | step 0050/60 | loss 0.1262 | acc 1.0000 | lr 0.000100
Epoch 007 | train: loss 0.1535, acc 0.9758 | lr 0.000100
Epoch 008 | step 0050/60 | loss 0.1299 | acc 1.0000 | lr 0.000100
Epoch 008 | train: loss 0.1296, acc 0.9812 | lr 0.000100
Epoch 009 | step

In [29]:
from src.models import BearingCNN1D

bio_model = BearingCNN1D(num_classes=3)
cfg = {
    'model': bio_model,                     # nn.Module já criado
    'train_loader': bio_loader,          # DataLoader
    'val_loader': None,              # ou None
    'epochs': 20,
    'optimizer': {'name': 'adam', 'lr': 1e-4, 'weight_decay': 1e-4},
    'scheduler': {'name': 'plateau', 'factor': 0.5, 'patience': 5, 'min_lr': 1e-6},
    'criterion': torch.nn.CrossEntropyLoss(),
    'amp': True,
    'grad_clip': 1.0,
    'early_stopping': {'patience': 30, 'min_delta': 0.0},
    'checkpoint_dir': 'checkpoints',
    'checkpoint_name': 'best_bio.pt',
    'save_every_epoch': False,
    'log_interval': 50,
    'eval_interval': 1,
}
history, best_path = train_model(cfg)

/tmp/ipykernel_3149089/1975858508.py:108: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp_enabled and device == 'cuda'))


Epoch 001 | train: loss 0.8961, acc 0.5281 | lr 0.000100
Epoch 002 | train: loss 0.6418, acc 0.8219 | lr 0.000100
Epoch 003 | train: loss 0.5611, acc 0.8785 | lr 0.000100
Epoch 004 | train: loss 0.4969, acc 0.9191 | lr 0.000100
Epoch 005 | train: loss 0.4407, acc 0.9319 | lr 0.000100
Epoch 006 | train: loss 0.4011, acc 0.9417 | lr 0.000100
Epoch 007 | train: loss 0.3664, acc 0.9510 | lr 0.000100
Epoch 008 | train: loss 0.3308, acc 0.9618 | lr 0.000100
Epoch 009 | train: loss 0.2982, acc 0.9698 | lr 0.000100
Epoch 010 | train: loss 0.2680, acc 0.9816 | lr 0.000100
Epoch 011 | train: loss 0.2513, acc 0.9812 | lr 0.000100
Epoch 012 | train: loss 0.2240, acc 0.9899 | lr 0.000100
Epoch 013 | train: loss 0.1998, acc 0.9896 | lr 0.000100
Epoch 014 | train: loss 0.1840, acc 0.9938 | lr 0.000100
Epoch 015 | train: loss 0.1724, acc 0.9927 | lr 0.000100
Epoch 016 | train: loss 0.1528, acc 0.9951 | lr 0.000100
Epoch 017 | train: loss 0.1421, acc 0.9965 | lr 0.000100
Epoch 018 | train: loss 0.1262,

## Validation

In [11]:
import torch
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

def validate_model(config):
    """
    Validate the model using the given configuration dictionary.
    Args:
        config (dict): Dictionary with parameters. Must include:
            - model: PyTorch model
            - val_loader: DataLoader for validation
            - criterion: Loss function
            - device: "cuda" or "cpu"
            - metrics (optional): dict with metric_name: function(y_true, y_pred)
    Returns:
        dict: validation results (loss and metrics)
    """
    model = config["model"]
    val_loader = config["val_loader"]
    criterion = config["criterion"]
    device = config.get("device", "cpu")
    metrics = config.get("metrics", {})

    model.eval()
    val_loss = 0.0
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)
            val_loss += loss.item() * x.size(0)

            preds = torch.argmax(outputs, dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    val_loss /= len(val_loader.dataset)

    results = {"val_loss": val_loss}
    results["accuracy"] = accuracy_score(y_true, y_pred)
    results["f1_macro"] = f1_score(y_true, y_pred, average="macro")
    results["f1_micro"] = f1_score(y_true, y_pred, average="micro")
    results["confusion_matrix"] = confusion_matrix(y_true, y_pred)

    # Extra metrics if user passed
    for name, func in metrics.items():
        results[name] = func(y_true, y_pred)

    return results


In [17]:
from torch.utils.data import DataLoader
# Dataset
root = "data/processed/uored_hierarchical/setup_1"
val_ds = BearingNPYDataset(f"{root}/bin_val", task="bin")
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

config_val = {
    "model": bin_model,
    "val_loader": val_loader,
    "criterion": torch.nn.CrossEntropyLoss(),
    "device": "cuda",
}

val_results = validate_model(config_val)
# print(val_results)
print(f"Accuracy: {val_results['accuracy']}")
print(val_results['confusion_matrix'])


Accuracy: 0.9291666666666667
[[172  68]
 [  0 720]]


In [35]:
# Exemplo
from src.models import BearingCNN1D

te_ds = BearingNPYDataset(f"{root}/test", task="bin")
te_loader = DataLoader(te_ds, batch_size=64, shuffle=False, num_workers=0)

binary_clf = bin_model
multi_clf = bio_model
hier_clf = HierarchicalClassifierProb(binary_clf, multi_clf, device="cuda")

config_val = {
    "model": hier_clf,
    "val_loader": te_loader,
    "criterion": torch.nn.NLLLoss(),
    "device": "cuda",
}

val_results = validate_model(config_val)
# print(val_results)
print(f"Accuracy: {val_results['accuracy']}")
print(val_results['confusion_matrix'])

Accuracy: 0.9666666666666667
[[232   8]
 [  0   0]]
